In [0]:
customers_silver = spark.table(
    "databricks_pipeline_copy_merge.broze.customers_bronze"
)

In [0]:
customers_silver.display()

In [0]:
%sql
create table databricks_pipeline_copy_merge.silver.customers_silver as select * from databricks_pipeline_copy_merge.broze.customers_bronze
where 1=2;

In [0]:
customers_silver.printSchema()

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
customers_silver = customers_silver.withColumn(
    "updated_at",
    F.to_timestamp("updated_at","dd/MM/yyyy HH:mm"))

In [0]:
customers_silver.printSchema()

In [0]:
%sql
DROP TABLE IF EXISTS databricks_pipeline_copy_merge.silver.customers_silver;

In [0]:
%sql
CREATE TABLE databricks_pipeline_copy_merge.silver.customers_silver
(
    customer_id INT,
    customer_name STRING,
    city STRING,
    state STRING,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
from delta.tables import DeltaTable

target_table = "databricks_pipeline_copy_merge.silver.customers_silver"

target = DeltaTable.forName(spark, target_table)

(
    target.alias("target")
    .merge(
        customers_silver.alias("source"),
        "target.customer_id = source.customer_id"
    )

    .whenMatchedUpdate(
        condition="source.updated_at > target.updated_at",
        set={
            "customer_name": "source.customer_name",
            "city": "source.city",
            "state": "source.state",
            "updated_at": "source.updated_at"
        }
    )

    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "customer_name": "source.customer_name",
            "city": "source.city",
            "state": "source.state",
            "updated_at": "source.updated_at"
        }
    )

    .execute()
)

In [0]:
%sql
select * from databricks_pipeline_copy_merge.silver.customers_silver;